# Tree-of-Thought এবং ReAct

দুইটা independent, সম্পূর্ণ self-contained demonstration:

**PART A — Tree-of-Thought (Yao et al., 2023) আসল search হিসেবে, metaphor নয়।**
Toy puzzle: একটা integer `start` থেকে শুরু করে একটা fixed operation set (`+3`, `-1`, `*2`) দিয়ে যত কম step-এ সম্ভব একটা integer `target`-এ পৌঁছাও। আমরা একই problem-এর ওপর তিনটা সত্যিকারের ভিন্ন search strategy implement করি এবং identical instances-এ তাদের তুলনা করি:

1. State deduplication সহ exhaustive BFS — true optimal (fewest-steps) solution length-এর ground-truth oracle।
2. Tree-of-Thought-ধাঁচের BEST-FIRST SEARCH WITH HEURISTIC PRUNING: প্রতিটি depth-এ current frontier-এর প্রতিটি child generate করো (branching factor 3, তাই একটা UNPRUNED tree 3^depth হারে বাড়ত — সত্যিকারের branch explosion), প্রতিটি candidate-কে heuristic (target থেকে distance) দিয়ে score করো, আর এগোনোর আগে শুধু সেরা `beam_width` টি candidate রাখো। এটাই ঠিক Yao et al.-এর রেসিপি: অনেকগুলো "thought" generate করো, একটা evaluator দিয়ে বিচার করাও, promising-গুলো রাখো, বাকিগুলো discard করো।
3. NAIVE GREEDY single-path search: কোনো branching-ই নেই, সবসময় একক locally-best next step-টা নাও, কখনো backtrack করো না। আমরা random puzzle instances-এর একটা batch-এর ওপর empirically দেখাই কতবার এটা revisit loop-এ আটকে যায় বা BFS ও pruned tree search যা খুঁজে পায় তার চেয়েও strictly worse (non-optimal) solution-এ পৌঁছায়।

**PART B — ReAct (Yao et al., 2022):** একটা ছোট্ট SCRIPTED agent loop (train করা model নয় — এখানে কোনো LLM নেই) যেটা একটা toy multi-step প্রশ্ন সমাধান করতে Thought / Action / Observation steps-কে interleave করে, যেখানে উত্তর দিতে সত্যিই বাইরের tools (একটা lookup table আর একটা calculator) call করতে হয়, আর প্রতিটি step-এ full trace print করে।

**Runtime:** CPU-তে 5 সেকেন্ডেরও কম (pure Python search + scripted logic, কোনো model training নেই)।

**চালানোর নিয়ম:**
- উপর থেকে নিচে cell-গুলো ক্রমান্বয়ে চালাও।
- আসল স্ক্রিপ্ট: `python example.py`

In [ ]:
import random
from collections import deque

random.seed(0)

## Part A — Tree-of-Thought, আসল search হিসেবে

Puzzle: fixed operation set (`+3`, `-1`, `*2`) দিয়ে যত কম step-এ সম্ভব target-এ পৌঁছানো। তিনটা strategy — BFS oracle (ground truth), ToT-ধাঁচের pruned beam search, আর naive greedy — identical instances-এ পাশাপাশি compare করা হয়। Cell-এর শেষে `part_a_demo()` চলে।

In [ ]:
# ===========================================================================
# PART A: TREE-OF-THOUGHT -- SEARCH হিসেবে
# ===========================================================================

OPS = [
    ("+3", lambda v: v + 3),
    ("-1", lambda v: v - 1),
    ("*2", lambda v: v * 2),
]
VALUE_BOUND = 500          # [-VALUE_BOUND, VALUE_BOUND]-এর বাইরের states "too far afield" হিসেবে prune হয়
MAX_DEPTH = 14


def in_bounds(v):
    return -VALUE_BOUND <= v <= VALUE_BOUND


def heuristic(value, target):
    """সহজ, সস্তা evaluator: target থেকে এই state-টা কত দূরে? (admissible
    হওয়ার দাবি করা হয়নি -- ঠিক সেই ধরণের মোটামুটি, শেখা-বা-হাতে-লেখা
    evaluator যেটা Tree-of-Thought candidate thoughts বিচার করতে ব্যবহার
    করে, optimality-র guarantee নয়।)"""
    return abs(value - target)


def bfs_optimal(start, target, max_depth=MAX_DEPTH):
    """Ground-truth oracle: DEDUPLICATED state graph-এর ওপর exhaustive
    breadth-first search। যেহেতু প্রতিটি edge-এর cost সমান (এক step), BFS
    প্রথমবার কোনো state-এ পৌঁছানো মানেই একটা shortest path -- এতে সবকিছু
    তুলনা করার মতো true optimal step count পাওয়া যায়।"""
    if start == target:
        return [], 0
    visited = {start}
    frontier = deque([(start, [])])
    nodes_visited = 0
    while frontier:
        value, path = frontier.popleft()
        for op_name, op_fn in OPS:
            new_value = op_fn(value)
            nodes_visited += 1
            if new_value == target:
                return path + [op_name], nodes_visited
            if in_bounds(new_value) and new_value not in visited and len(path) + 1 < max_depth:
                visited.add(new_value)
                frontier.append((new_value, path + [op_name]))
    return None, nodes_visited


def tot_beam_search(start, target, beam_width, max_depth=MAX_DEPTH):
    """Tree-of-Thought-ধাঁচের search: প্রতিটি depth-এ current frontier-এর
    সব child expand করো (branching factor 3 -- এটাই raw, unpruned tree),
    তারপর পরের depth-এ যাওয়ার আগে heuristic score-এ সবচেয়ে ভালো (সবচেয়ে
    কম) `beam_width` টি child-ই রাখো। পাওয়া path, GENERATED node-র সংখ্যা
    (অর্থাৎ raw unpruned tree তখন এই বিন্দুতে কত বড়ো হতো), আর pruning-এর
    পরে সত্যিই KEPT/explored node-র সংখ্যা return করে।"""
    if start == target:
        return [], 0, 0
    frontier = [(heuristic(start, target), start, [])]
    total_generated = 0
    total_kept = 1
    for depth in range(max_depth):
        candidates = []
        for _, value, path in frontier:
            for op_name, op_fn in OPS:
                new_value = op_fn(value)
                total_generated += 1
                if new_value == target:
                    return path + [op_name], total_generated, total_kept
                if in_bounds(new_value):
                    candidates.append((heuristic(new_value, target), new_value, path + [op_name]))
        if not candidates:
            return None, total_generated, total_kept
        candidates.sort(key=lambda c: c[0])
        frontier = candidates[:beam_width]
        total_kept += len(frontier)
    return None, total_generated, total_kept


def greedy_single_path(start, target, max_steps=MAX_DEPTH):
    """Naive greedy: ONE path, কোনো branching নেই, কোনো backtracking নেই।
    প্রতিটি step-এ সেই single operation-টা নাও যেটা target-এর heuristic
    distance সবচেয়ে কমায় (fixed tie-break order: +3, -1, *2)। বেছে নেওয়া
    পরের state-টা যদি এই একই path-এ আগে visited হয়ে থাকে, তবে greedy একটা
    loop-এ আটকে যায় যেখান থেকে বের হওয়া অসম্ভব (অন্য কিছু চেষ্টা করার
    mechanism ওর নেই) আর আমরা চিরকাল ঘোরার বদলে failure report করি।"""
    value = start
    path = []
    visited_on_path = {start}
    for _ in range(max_steps):
        if value == target:
            return path, "success"
        best = min(
            ((heuristic(op_fn(value), target), op_name, op_fn(value)) for op_name, op_fn in OPS),
            key=lambda c: c[0],
        )
        _, op_name, new_value = best
        if new_value == target:
            return path + [op_name], "success"
        if new_value in visited_on_path or not in_bounds(new_value):
            return path, "stuck (revisited a state / left bounds, no backtracking available)"
        visited_on_path.add(new_value)
        path.append(op_name)
        value = new_value
    return path, "gave up (exceeded max steps without reaching target)"


def part_a_demo():
    print("=" * 78)
    print("PART A: TREE-OF-THOUGHT AS REAL SEARCH")
    print("=" * 78)
    print(f"Puzzle: reach `target` from `start` using ops {[o[0] for o in OPS]}, fewest steps.")
    print(f"State bound: |value| <= {VALUE_BOUND}. Max depth: {MAX_DEPTH}.\n")

    print("-" * 78)
    print("One concrete instance, all three strategies side by side")
    print("-" * 78)
    start, target = 0, 35
    opt_path, bfs_nodes = bfs_optimal(start, target)
    beam_path, beam_generated, beam_kept = tot_beam_search(start, target, beam_width=3)
    greedy_path, greedy_status = greedy_single_path(start, target)
    print(f"start={start}, target={target}")
    print(f"  BFS oracle (exhaustive, deduplicated): {len(opt_path)} steps  {opt_path}")
    print(f"    nodes visited: {bfs_nodes}")
    print(f"  ToT beam search (beam_width=3):        {len(beam_path)} steps  {beam_path}")
    print(f"    raw tree nodes generated: {beam_generated}   nodes kept after pruning: {beam_kept}")
    unpruned_tree_size = sum(3 ** d for d in range(1, len(beam_path) + 1))
    print(f"    (an UNPRUNED tree exploring every branch to this depth would have")
    print(f"     generated {unpruned_tree_size} nodes -- pruning kept only {beam_kept}.)")
    print(f"  Naive greedy (single path, no backtracking): status = {greedy_status}")
    print(f"    path so far: {greedy_path}")

    print("\n" + "-" * 78)
    print("Batch comparison across 200 random puzzle instances")
    print("-" * 78)
    print("(start in [0,15], target in [-10,60], fixed random seed)\n")

    num_instances = 200
    greedy_fail_count = 0
    greedy_suboptimal_count = 0
    beam_matches_optimal_count = 0
    total_bfs_nodes = 0
    total_beam_generated = 0
    total_beam_kept = 0
    first_failure_example = None

    for _ in range(num_instances):
        s = random.randint(0, 15)
        t = random.randint(-10, 60)
        opt_path, bfs_nodes = bfs_optimal(s, t)
        if opt_path is None:
            continue  # আমাদের search bound/depth-এর বাইরের বিরল instance-টা একদম skip করো
        beam_path, beam_generated, beam_kept = tot_beam_search(s, t, beam_width=3)
        g_path, g_status = greedy_single_path(s, t)

        total_bfs_nodes += bfs_nodes
        total_beam_generated += beam_generated
        total_beam_kept += beam_kept

        if beam_path is not None and len(beam_path) == len(opt_path):
            beam_matches_optimal_count += 1

        if g_status != "success":
            greedy_fail_count += 1
            if first_failure_example is None:
                first_failure_example = (s, t, opt_path, g_path, g_status)
        elif len(g_path) > len(opt_path):
            greedy_suboptimal_count += 1
            if first_failure_example is None:
                first_failure_example = (s, t, opt_path, g_path, "succeeded but suboptimal")

    print(f"{'strategy':40s}{'result':>36}")
    print(f"{'BFS oracle: always optimal':40s}{'100% (by construction)':>36}")
    print(f"{'ToT beam search matches optimal length':40s}{beam_matches_optimal_count}/{num_instances:>4}"
          f" = {100*beam_matches_optimal_count/num_instances:.1f}%")
    fails_or_worse = greedy_fail_count + greedy_suboptimal_count
    print(f"{'Greedy fails OR is strictly suboptimal':40s}{fails_or_worse}/{num_instances:>4}"
          f" = {100*fails_or_worse/num_instances:.1f}%")
    print(f"  (of which outright stuck/gave up: {greedy_fail_count}, "
          f"succeeded but took more steps than optimal: {greedy_suboptimal_count})")

    print(f"\nAverage nodes examined per instance:")
    print(f"  BFS oracle (exhaustive):        {total_bfs_nodes/num_instances:.1f}")
    print(f"  ToT beam search -- generated:   {total_beam_generated/num_instances:.1f}"
          f"  (raw tree nodes proposed, pre-pruning)")
    print(f"  ToT beam search -- kept:        {total_beam_kept/num_instances:.1f}"
          f"  (nodes actually carried forward after pruning)")
    print(f"  Greedy (single path):           1 node examined per step, by definition")

    if first_failure_example:
        s, t, opt_path, g_path, g_status = first_failure_example
        print(f"\nConcrete failure case: start={s}, target={t}")
        print(f"  Optimal (BFS):  {len(opt_path)} steps -- {opt_path}")
        print(f"  Greedy result:  {g_status}, path so far {g_path} ({len(g_path)} steps)")

    print(f"\n-> Across {num_instances} random instances, ToT-style beam search matches the")
    print(f"   true optimal path length {100*beam_matches_optimal_count/num_instances:.1f}% of the time while examining only")
    print(f"   {total_beam_kept/num_instances:.1f} nodes on average per instance (vs {total_bfs_nodes/num_instances:.1f} for exhaustive BFS) --")
    print(f"   pruning the raw {total_beam_generated/num_instances:.1f}-node branching tree down to a fraction of its size.")
    print(f"   Naive greedy, with no ability to branch or backtrack, fails outright or")
    print(f"   ends up with a worse-than-optimal solution in {100*fails_or_worse/num_instances:.1f}% of instances -- exactly the")
    print("   failure mode Tree-of-Thought is designed to avoid: a single locally-best")
    print("   choice at one step can commit you to a state with no good continuation,")
    print("   whereas keeping several candidate branches alive (even a small beam)")
    print("   lets the search recover from a locally-tempting but globally bad move.")
part_a_demo()

## Part B — ReAct: Reason + Act, পর্যায়ক্রমে

একটা scripted ReAct-ধাঁচের agent — Thought steps-গুলো লিখিত Python string (কোনো language model নেই), কিন্তু tool execution সত্যিকারের: `lookup_capital`, `lookup_population` আর `calculator` হলো আসল Python function, যারা calling code-এর আগে-থেকে-জানা-না-থাকা real value return করে। Cell-এর শেষে `part_b_demo()` সেই agent-কে চালায় — সফল পথ এবং lookup-এ অনুপস্থিত দেশের failure-path দুটোই।

In [ ]:
# ===========================================================================
# PART B: REACT -- REASONING আর ACTING, পর্যায়ক্রমে (INTERLEAVED)
# ===========================================================================

# একটা ছোট্ট "world" যেটা toy agent query করতে পারে -- বাস্তব tools/APIs-এর প্রতিনিধি।
CAPITAL_LOOKUP = {
    "France": "Paris",
    "Japan": "Tokyo",
}
POPULATION_LOOKUP = {   # আনুমানিক, শুধু demo-র জন্য
    "Paris": 2_148_000,
    "Tokyo": 13_960_000,
}


def tool_lookup_capital(country):
    return CAPITAL_LOOKUP.get(country, "UNKNOWN")


def tool_lookup_population(city):
    return POPULATION_LOOKUP.get(city, None)


def tool_calculator(expression):
    """একটা বাস্তব calculator tool -- একটা restricted arithmetic
    expression string evaluate করে। এটা একটা genuine function call যার
    genuine return value আছে, কোনো fake API stub নয়।"""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError(f"Unsupported characters in expression: {expression!r}")
    return eval(expression, {"__builtins__": {}}, {})


def react_agent(country):
    """একটা SCRIPTED ReAct-ধাঁচের loop (Yao et al., 2022): agent-র Thought
    steps এখানে নির্দিষ্ট (কোনো LLM সেগুলো তৈরি করছে না -- Part B-তে এই
    file-এ কোনো model-ই নেই), কিন্তু STRUCTURE -- পর্যায়ক্রমে Thought,
    Action (একটা real tool call), আর Observation (tool-এর real return
    value), প্রতিটা Observation-কে পরের Thought-এ ফিড করা -- ঠিক সেই ReAct
    interaction pattern, এমন একটা task-এ প্রয়োগ করা যার উত্তর দিতে সত্যিই
    দুইটা chained tool call আর arithmetic দরকার।"""
    print(f"\nTask: \"What is the population of {country}'s capital city, divided by 1000,")
    print("       rounded to the nearest whole number?\"\n")

    print("Thought 1: I don't know the capital of this country. I need to look it up.")
    print(f"Action 1: lookup_capital(country={country!r})")
    capital = tool_lookup_capital(country)
    print(f"Observation 1: {capital!r}")
    if capital == "UNKNOWN":
        print("Thought: the lookup tool has no entry for this country. Stopping.")
        return None

    print(f"\nThought 2: Now I know the capital is {capital}. I need its population.")
    print(f"Action 2: lookup_population(city={capital!r})")
    population = tool_lookup_population(capital)
    print(f"Observation 2: {population!r}")
    if population is None:
        print("Thought: the lookup tool has no population entry for this city. Stopping.")
        return None

    print(f"\nThought 3: I have the population ({population}). The question asks for it")
    print("           divided by 1000 and rounded. I should use the calculator, not")
    print("           do the arithmetic myself, to guarantee the result is exact.")
    expr = f"{population} / 1000"
    print(f"Action 3: calculator(expression={expr!r})")
    raw_result = tool_calculator(expr)
    print(f"Observation 3: {raw_result!r}")

    final_answer = round(raw_result)
    print(f"\nThought 4: {raw_result} rounds to {final_answer}. I have everything needed to answer.")
    print(f"Final Answer: {final_answer}")
    return final_answer


def part_b_demo():
    print("\n" + "=" * 78)
    print("PART B: REACT -- A SCRIPTED REASON+ACT+OBSERVE LOOP")
    print("=" * 78)
    print("This agent's 'thoughts' are scripted Python strings (there is no language")
    print("model anywhere in this section) -- what's being demonstrated is the REACT")
    print("interaction PATTERN itself: Thought -> Action (real tool call) -> Observation")
    print("-> next Thought, chained until the task is answered, with each tool call")
    print("returning a REAL value that genuinely changes what happens next.\n")

    answer = react_agent("France")
    expected = round(POPULATION_LOOKUP["Paris"] / 1000)
    print(f"\n-> Verification: population of Paris ({POPULATION_LOOKUP['Paris']}) / 1000, rounded,")
    print(f"   is independently computed as {expected}. The agent's final answer ({answer}) matches:")
    print(f"   {answer == expected}. Two real, distinct tool calls (a lookup, then a")
    print("   calculator call) were required and executed to reach it -- neither tool's")
    print("   result was known to the agent in advance, exactly the point of Acting")
    print("   (gathering real information) interleaved with Reasoning (deciding what")
    print("   to do with it), rather than trying to answer from the question text alone.")

    print("\nRunning the same agent on a country with no lookup data on record, to show")
    print("the loop stopping cleanly on a real 'observation says I can't proceed' case:")
    react_agent("Germany")
part_b_demo()

## পুরো demonstration চালানো

`main()` function-টা `part_a_demo()` আর `part_b_demo()` দুটোই call করে। শেষ cell-এ `main()` কল হয়। (উপরের section cells-গুলোতে demo-গুলো ইতিমধ্যে run হয়ে গেছে, তাই `main()` চালানোর ফলে আউটপুট দ্বিগুণ print হবে — এটাই প্রত্যাশিত।)

In [ ]:
def main():
    part_a_demo()
    part_b_demo()

In [ ]:
main()